# xp-06 — Label-cutoff sensitivity sweep

Experiment notebook: how sensitive is the rule-vs-model comparison to the **label definition**?
This notebook changes nothing in the main line (w04–w06 stay at the committed 0.1pp cutoff).

Pre-registered plan (decided before running):
- Same w05 load query (120,258 pages, same rows/order) so the rule must reproduce 30% / 44%
  and LR ~50% / 32% at cutoff 0.1.
- Same 10 features, same client-holdout split (GroupShuffleSplit, test_size 0.2, seed 42).
- Sweep the label cutoff: 0.0, 0.05, 0.1, 0.2, 0.3, 0.5 percentage points.
  `below_tier_outcome = (gap_label > cutoff)`. Each cutoff re-trains LR and re-grades the rule.
- Per cutoff report: train base rate, test base rate, rule p@10/@50, LR p@10/@50, LR AUPRC,
  lift vs test base rate.
- Verdict is descriptive ("is the comparison stable across the label definition?").

Guardrails:
- No tuning on the holdout. The sweep is descriptive; no cutoff is selected by best test precision.
- Changing the cutoff redefines "below tier" (the w03 contract). If a cutoff were ever adopted,
  it must be justified on train / inner split only.
- Writes work/outputs/threshold_sweep.csv and changes no other notebook.


In [8]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Same w05 load (same rows, same order => the rule must reproduce 30/44 at cutoff 0.1).
data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']
raw_gap_label = data['gap_label'].copy()

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

print(f'Pages in data: {len(data):,}  Clients: {data["client_hash_id"].nunique():,}')
print('gap_label (raw, pre-fill) summary: '
      f'min {raw_gap_label.min():.3f}, median {raw_gap_label.median():.3f}, max {raw_gap_label.max():.3f}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Pages in data: 120,258  Clients: 47
gap_label (raw, pre-fill) summary: min -99.673, median 0.149, max 0.406


In [9]:
from sklearn.model_selection import GroupShuffleSplit

sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sp.split(data, groups=data['client_hash_id']))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

print(f'Train: {len(train):,} pages from {train["client_hash_id"].nunique()} clients')
print(f'Test:  {len(test):,} pages from {test["client_hash_id"].nunique()} clients')


Train: 112,968 pages from 37 clients
Test:  7,290 pages from 10 clients


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score

num_features = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
                'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_features = ['content_type', 'main_intent', 'position_tier']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])
X_train = preprocessor.fit_transform(train[num_features + cat_features])

def precision_at_k(score, y, k):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

CUTOFFS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]

has_volume = (test['impressions_fw'] >= 500).astype(int)
ctr_gap = test['tier_ctr_gap'].clip(lower=0)
rule_score = pd.Series((has_volume * ctr_gap * test['impressions_fw']).values, index=test.index)

rows = []
for cutoff in CUTOFFS:
    y_full = (raw_gap_label > cutoff).astype(int)
    y_train = y_full.loc[train.index]
    y_test = y_full.loc[test.index]

    row = {
        'cutoff_pp': cutoff,
        'train_base_rate': y_train.mean(),
        'test_base_rate': y_test.mean(),
        'rule_p10': precision_at_k(rule_score, y_test, 10),
        'rule_p50': precision_at_k(rule_score, y_test, 50),
    }

    if y_train.nunique() < 2 or y_test.nunique() < 2:
        row.update({'lr_p10': np.nan, 'lr_p50': np.nan, 'lr_auprc': np.nan,
                    'note': 'single-class train/test, LR skipped'})
    else:
        lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
        lr.fit(X_train, y_train)
        lr_prob = pd.Series(
            lr.predict_proba(preprocessor.transform(test[num_features + cat_features]))[:, 1],
            index=test.index
        )
        row.update({'lr_p10': precision_at_k(lr_prob, y_test, 10),
                    'lr_p50': precision_at_k(lr_prob, y_test, 50),
                    'lr_auprc': average_precision_score(y_test, lr_prob),
                    'note': ''})

    rows.append(row)

sweep = pd.DataFrame(rows)
sweep['rule_lift50'] = sweep['rule_p50'] / sweep['test_base_rate']
sweep['lr_lift50'] = sweep['lr_p50'] / sweep['test_base_rate']

out_dir = Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'threshold_sweep.csv'
sweep.to_csv(out_path, index=False)

print('Label-cutoff sweep (seed 42 client holdout)')
print()
print(sweep.round(4).to_string(index=False))
print()
print(f'Wrote {out_path}')
print()
print('Sanity: cutoff 0.1 should reproduce w05: rule 30% / 44%, LR 50% / 32%, test base rate 15.9%.')


Label-cutoff sweep (seed 42 client holdout)

 cutoff_pp  train_base_rate  test_base_rate  rule_p10  rule_p50  lr_p10  lr_p50  lr_auprc                                note  rule_lift50  lr_lift50
      0.00           0.7255          0.1982       0.4      0.48     0.0    0.02    0.2055                                           2.4216     0.1009
      0.05           0.6534          0.1705       0.4      0.48     0.3    0.22    0.2207                                           2.8151     1.2903
      0.10           0.6023          0.1590       0.3      0.44     0.5    0.32    0.2350                                           2.7676     2.0128
      0.20           0.3854          0.1181       0.3      0.36     1.0    0.84    0.4073                                           3.0481     7.1122
      0.30           0.1852          0.0748       0.1      0.14     0.9    0.90    0.5276                                           1.8727    12.0385
      0.50           0.0000          0.0000       0.0  

In [11]:
# How many pages flip label between adjacent cutoffs, and the base-rate curve.
print('Label flips between adjacent cutoffs (whole 120,258 pages):')
for i in range(len(CUTOFFS) - 1):
    a = (raw_gap_label > CUTOFFS[i]).astype(int)
    b = (raw_gap_label > CUTOFFS[i + 1]).astype(int)
    flips = (a != b).sum()
    print(f'  {CUTOFFS[i]:>4} -> {CUTOFFS[i + 1]:>4}: {flips:>6,} pages flip ({flips / len(data):.1%})')

print()
print('Test base rate vs cutoff (the floor each score must beat), precision@50:')
for r in sweep.itertuples():
    print(f'  cutoff {r.cutoff_pp:>4}: base {r.test_base_rate:.1%} | '
          f'rule {r.rule_p50:.1%} (x{r.rule_lift50:.1f}) | '
          f'LR {r.lr_p50:.1%} (x{r.lr_lift50:.1f})')


Label flips between adjacent cutoffs (whole 120,258 pages):
   0.0 -> 0.05:  8,346 pages flip (6.9%)
  0.05 ->  0.1:  5,859 pages flip (4.9%)
   0.1 ->  0.2: 24,797 pages flip (20.6%)
   0.2 ->  0.3: 22,933 pages flip (19.1%)
   0.3 ->  0.5: 21,466 pages flip (17.8%)

Test base rate vs cutoff (the floor each score must beat), precision@50:
  cutoff  0.0: base 19.8% | rule 48.0% (x2.4) | LR 2.0% (x0.1)
  cutoff 0.05: base 17.1% | rule 48.0% (x2.8) | LR 22.0% (x1.3)
  cutoff  0.1: base 15.9% | rule 44.0% (x2.8) | LR 32.0% (x2.0)
  cutoff  0.2: base 11.8% | rule 36.0% (x3.0) | LR 84.0% (x7.1)
  cutoff  0.3: base 7.5% | rule 14.0% (x1.9) | LR 90.0% (x12.0)
  cutoff  0.5: base 0.0% | rule 0.0% (xnan) | LR nan% (xnan)


## Seed stability check (pre-registered)

The sweep above is seed 42 only. This section repeats the cutoff sweep on the other three
client-holdout seeds (43, 44, 45) and reports how stable the rule-vs-LR comparison is.

Pre-registered criterion (decided before running):
- The strict-cutoff LR lead is CONFIRMED only if LR p@50 beats the rule on >= 3 of 4 seeds
  at that cutoff.
- No cutoff is adopted from this sweep. Any cutoff change must be justified on train / inner
  split only (experiment rules). The committed 0.1pp definition stays the main line.


In [12]:
from sklearn.model_selection import GroupShuffleSplit

def client_split(df, seed):
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(sp.split(df, groups=df['client_hash_id']))
    return df.iloc[tr].copy(), df.iloc[te].copy()

SEEDS = [42, 43, 44, 45]
rows = []
for seed in SEEDS:
    tr, te = client_split(data, seed)
    pre = ColumnTransformer([
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])
    X_tr = pre.fit_transform(tr[num_features + cat_features])
    rule_te = pd.Series(
        ((te['impressions_fw'] >= 500).astype(int) * te['tier_ctr_gap'].clip(lower=0) * te['impressions_fw']).values,
        index=te.index
    )
    for cutoff in CUTOFFS:
        y_full = (raw_gap_label > cutoff).astype(int)
        y_tr = y_full.loc[tr.index]
        y_te = y_full.loc[te.index]
        row = {
            'seed': seed,
            'cutoff_pp': cutoff,
            'train_base_rate': y_tr.mean(),
            'test_base_rate': y_te.mean(),
            'rule_p10': precision_at_k(rule_te, y_te, 10),
            'rule_p50': precision_at_k(rule_te, y_te, 50),
        }
        if y_tr.nunique() < 2 or y_te.nunique() < 2:
            row.update({'lr_p10': np.nan, 'lr_p50': np.nan, 'lr_auprc': np.nan,
                        'note': 'single-class train/test, LR skipped'})
        else:
            lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
            lr.fit(X_tr, y_tr)
            lr_prob = pd.Series(
                lr.predict_proba(pre.transform(te[num_features + cat_features]))[:, 1],
                index=te.index
            )
            row.update({'lr_p10': precision_at_k(lr_prob, y_te, 10),
                        'lr_p50': precision_at_k(lr_prob, y_te, 50),
                        'lr_auprc': average_precision_score(y_te, lr_prob),
                        'note': ''})
        rows.append(row)

seed_sweep = pd.DataFrame(rows)
print(f'Seed sweep rows: {len(seed_sweep)} ({len(SEEDS)} seeds x {len(CUTOFFS)} cutoffs)')


Seed sweep rows: 24 (4 seeds x 6 cutoffs)


In [13]:
out_path_seeds = out_dir / 'threshold_sweep_seeds.csv'
seed_sweep.to_csv(out_path_seeds, index=False)

print('Per-seed sweep written to', out_path_seeds)
print()
print('Summary across seeds (client-holdout splits 42-45), precision@50 means:')
summary = seed_sweep.groupby('cutoff_pp').agg(
    mean_test_base=('test_base_rate', 'mean'),
    mean_rule_p10=('rule_p10', 'mean'),
    mean_rule_p50=('rule_p50', 'mean'),
    mean_lr_p10=('lr_p10', 'mean'),
    mean_lr_p50=('lr_p50', 'mean'),
).round(4)
print(summary.to_string())
print()
print('Seeds where LR p@50 beats the rule, per cutoff (>=3 of 4 = confirmed lead):')
for cutoff in CUTOFFS:
    sub = seed_sweep[seed_sweep['cutoff_pp'] == cutoff]
    wins = int((sub['lr_p50'] > sub['rule_p50']).sum())
    total = int(sub['lr_p50'].notna().sum())
    print(f'  cutoff {cutoff:>4}: LR wins p@50 in {wins}/{total} seeds')


Per-seed sweep written to ../../work/outputs/threshold_sweep_seeds.csv

Summary across seeds (client-holdout splits 42-45), precision@50 means:
           mean_test_base  mean_rule_p10  mean_rule_p50  mean_lr_p10  mean_lr_p50
cutoff_pp                                                                        
0.00               0.6034          0.850          0.860        0.750        0.755
0.05               0.5533          0.850          0.855        0.825        0.805
0.10               0.5130          0.775          0.810        0.875        0.830
0.20               0.3501          0.550          0.580        1.000        0.955
0.30               0.1875          0.175          0.170        0.975        0.965
0.50               0.0000          0.000          0.000          NaN          NaN

Seeds where LR p@50 beats the rule, per cutoff (>=3 of 4 = confirmed lead):
  cutoff  0.0: LR wins p@50 in 2/4 seeds
  cutoff 0.05: LR wins p@50 in 2/4 seeds
  cutoff  0.1: LR wins p@50 in 3/4 seeds


## Verdict (filled after the seed-42 + seeds 42-45 runs)

- [x] The rule-vs-LR comparison flips with the label definition: the rule wins loose
      cutoffs (0.0-0.1), LR dominates strict cutoffs (0.2-0.3), and at 0.5 the label
      is empty (max gap 0.406pp < 0.5pp).
- [x] The comparison is not stable across client-holdout splits either. Across seeds
      42-45, LR wins precision@50 in 3/4 seeds at the committed 0.1pp cutoff and 4/4
      at 0.2-0.3pp; the rule wins only the loose cutoffs (2/4 seeds). The seed-42
      "rule holds the wider cut" (44 vs 32) headline does not replicate.
- [x] The test base-rate floor drops as the cutoff tightens (seed 42: 19.8% -> 7.5%;
      mean across seeds: 60.3% -> 18.8%) and is 0% at 0.5. At 0.3 the rule falls
      below its base-rate floor on average.
- [x] Sanity: cutoff 0.1 on seed 42 reproduces w05 exactly (rule 30/44, LR 50/32,
      test base rate 15.9%).
- [x] No cutoff is adopted from this descriptive sweep. Choosing one from test
      precision would be holdout-tuned selection; the committed 0.1pp definition
      (the w03 contract) stays the main line.
- [x] Follow-up: the strict-cutoff LR lead was tested pre-registered in
      xp_pre_registered_cutoff.ipynb (xp-07) on a fresh, never-touched client
      holdout (seed 2026), with the cutoff chosen on train only (0.30pp, strictest
      with >=10% coverage). xp-07 CONFIRMED LR: p@10 80% vs rule 30%, p@50 92% vs
      rule 22%, base 18.6%, lift50 x4.96 vs x1.18. LR is adopted as the validated
      model at the 0.30pp definition; the committed 0.1pp rule remains the
      transparent baseline.
